# Enabling Validation for Robust Few-Shot Recognition

This notebook demonstrates how to perform **Validation-Enabled Stage-wise Tuning (VEST)** on DINOv2 for few-shot learning and evaluate the model on both ID and OOD datasets. Results are comparable to Table 4 in the paper.
1. model
    - [DINOv2 ViT-B/14 distilled with registers](https://github.com/facebookresearch/dinov2)
2. few-shot setting
    - 16 shots
3. PFT setting
    - top-1 blocks
4. dataset
    - ImageNet-1k (as ID dataset)
    - ImageNet-V2 (as OOD dataset)
    - ImageNet-S (as OOD dataset)
    - ImageNet-A (as OOD dataset)
    - ImageNet-R (as OOD dataset)

In [1]:
import torch
import numpy as np
import random

# Set the random seed for reproducibility
training_seed = 1
data_seed = 1

random.seed(data_seed)
np.random.seed(training_seed)
torch.manual_seed(training_seed)
torch.cuda.manual_seed_all(training_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## VEST-stage1: partial finetuning with retrieval augmentation

1. Load the DINOv2 model and prepare the dataset

In [7]:
from torchvision import transforms
from utils.models import DinoVisionTransformer_v2
import datasets


# Load the CLIP ViT-B/16 model
model = DinoVisionTransformer_v2(model_cfg='dinov2_vitb14_reg')
train_preprocess = transforms.Compose([
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
test_preprocess = transforms.Compose([
            transforms.Resize(224),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

# Prepare dataset and dataloader
root =  # Path to the ImageNet dataset
retrieved_root =  # Path to the retrieved dataset
num_shots = 16 # Number of shots for few-shot learning, options: 4, 8, 16
imagenet_train, text_name = datasets.build_imagenet_few_shot_dataset_demo('imagenet', 'train', data_seed, train_preprocess,
                                                                     root=root, num_shots=num_shots, w_retrival=True) # set w_retrival=True to use retrieval augmentation

# ID testset
imagenet_test, _ = datasets.build_imagenet_dataset('imagenet', 'test', test_preprocess, root=root)
# OOD testsets
imagenet_a_test, _ = datasets.build_imagenet_dataset('imagenet_a', 'test', test_preprocess, root=root)
imagenet_r_test, _ = datasets.build_imagenet_dataset('imagenet_r', 'test', test_preprocess, root=root)
imagenet_sketch_test, _ = datasets.build_imagenet_dataset('imagenet_sketch', 'test', test_preprocess, root=root)    
imagenetv2_test, _ = datasets.build_imagenet_dataset('imagenetv2', 'test', test_preprocess, root=root)
# validation set
valset_ID, _ = datasets.build_imagenet_few_shot_dataset_demo('imagenet', 'train', data_seed, test_preprocess, root=root, num_shots=num_shots)
valset_RT = datasets.build_validation_set_demo('retrieved', retrieved_root, data_seed, test_preprocess)

batch_size = 64
train_dataloader = torch.utils.data.DataLoader(
    dataset=imagenet_train,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=False)
val_dataloader_ID = torch.utils.data.DataLoader(
    dataset=valset_ID,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False)
val_dataloader_RT = torch.utils.data.DataLoader(
    dataset=valset_RT,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False)
val_dataloader = (val_dataloader_ID, val_dataloader_RT)

len(imagenet_train), len(valset_ID), len(valset_RT), len(imagenet_test), len(imagenet_a_test), len(imagenet_r_test), len(imagenet_sketch_test), len(imagenetv2_test)

Loading few-shot data from data_resource/imagenet/fewshot16_seed1.txt.
Loading retrieved data from data_resource/retrieved/T2T500.txt.
Loading few-shot data from data_resource/imagenet/fewshot16_seed1.txt.
Loading OOD validation data from data_resource/retrieved/val_seed1_retrieved.txt.


(487876, 16000, 45048, 50000, 7500, 30000, 50889, 10000)

2. Frozen the dinov2 model except the top-X blocks of visual encoder.

In [8]:
def frozen(model, ft_topk_blks):
    for param in model.parameters():
        param.requires_grad = False

    if ft_topk_blks == -1:
        print('Finetune all blocks of the visual transformer.')
        for param in model.parameters():
            param.requires_grad = True
    else:
        print(f'Finetune top-{ft_topk_blks} blocks of the visual transformer.')
        for blk in model.transformer.blocks[-ft_topk_blks:]:
            for param in blk.parameters():
                param.requires_grad = True

        for param in model.transformer.norm.parameters():
            param.requires_grad = True

# In our experiments, we adopt PFT on the top-1 blocks on DINOv2.
ft_topk_blks = 1
frozen(model, ft_topk_blks)

# double check the parameters
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

Finetune top-1 blocks of the visual transformer.
transformer.blocks.11.norm1.weight
transformer.blocks.11.norm1.bias
transformer.blocks.11.attn.qkv.weight
transformer.blocks.11.attn.qkv.bias
transformer.blocks.11.attn.proj.weight
transformer.blocks.11.attn.proj.bias
transformer.blocks.11.ls1.gamma
transformer.blocks.11.norm2.weight
transformer.blocks.11.norm2.bias
transformer.blocks.11.mlp.fc1.weight
transformer.blocks.11.mlp.fc1.bias
transformer.blocks.11.mlp.fc2.weight
transformer.blocks.11.mlp.fc2.bias
transformer.blocks.11.ls2.gamma
transformer.norm.weight
transformer.norm.bias


3. Randomly initialize the classifier.

In [9]:
import torch.nn as nn
class MyLinear(nn.Module):
    def __init__(self, input_dim=512, num_classes=1000, bias = False):
        super(MyLinear, self).__init__()

        self.linear = nn.Linear(input_dim, num_classes, bias=bias)
        self.num_classes = num_classes

    def forward(self, x):
        x = self.linear(x)

        return x

    def _init_weights(self, weights):
        # Initialize the weights of the linear layer with the given weights
        self.linear.weight = nn.Parameter(weights.clone())

In [10]:
# Set classifier
num_classes = 1000  # Number of classes
num_features = 768  # Number of features

classifier = MyLinear(input_dim=num_features, num_classes=num_classes, bias=False)

4. Define the optimizer and learning rate scheduler

In [11]:
from utils.scheduler import build_lr_scheduler

lr_backbone = 1e-6
lr_cls = 1e-4   # set the learning rate for the classifier
weight_decay = 0.01

# Define the optimizer
param_groups = [
            {"params": [p for name, p in model.named_parameters() if p.requires_grad], "lr": lr_backbone},
            {"params": [p for p in classifier.parameters()], "lr": lr_cls},
        ]
optimizer = torch.optim.AdamW(param_groups, lr=lr_cls, weight_decay=weight_decay, betas=(0.9, 0.999))

# Define the learning rate scheduler
num_epochs = 10
total_iter = len(train_dataloader) * num_epochs
warmup_iter = 18
warmup_lr = 1e-8
scheduler = build_lr_scheduler(optimizer,
                               lr_scheduler="cosine",
                               warmup_iter=warmup_iter,
                               max_iter=total_iter,
                               warmup_type="linear",
                               warmup_lr=warmup_lr,
                               verbose=False)

5. Start training

In [15]:
def test(dataloader, model, classifier, test_label_map=None, device='cuda'):
    model.eval()
    classifier.eval()
    with torch.no_grad():
        targets_list = []
        preds_list = []
        for idx, (inputs, targets) in enumerate(dataloader):
            inputs = inputs.to(device)
            targets = targets.to(device)

            image_features = model(inputs)
            outputs = classifier(image_features)
            outputs = outputs[:,test_label_map] # map the logits to the test dataset, primarily for ImageNet-A and ImageNet-R

            targets_list.append(targets.detach().cpu().numpy())
            preds_list.append(outputs.detach().cpu().numpy())

    targets_list = np.hstack(targets_list)
    preds_list = np.vstack(preds_list)
    preds_list = torch.tensor(preds_list)
    test_acc = (torch.softmax(preds_list, dim=1).argmax(1).numpy() == targets_list).mean()

    model.train()
    classifier.train()

    return test_acc*100

In [ ]:
import tqdm
import torchmetrics

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
CE_criterion = nn.CrossEntropyLoss()

best_val_acc = -1
loss, acc = [-1] * 2
val_acc = (-1, -1)
val_acc_list_ID = []
val_acc_list_RT = []
ckpt_path = 'demo_ckpt/VEST_dinov2_top4/' # path to save checkpoints
num_iter = 0

model.to(device)
classifier.to(device)
model.train()
classifier.train()
print(f"Start few-shot finetuning with RA ......")
for epoch in range(1, num_epochs + 1):

    train_acc = torchmetrics.Accuracy(num_classes=num_classes, task="multiclass", top_k=1)
    train_acc.to(device)

    pbar_iter = tqdm.tqdm(train_dataloader)
    for idx, (images, targets) in enumerate(pbar_iter):
        num_iter += 1
        pbar_iter.set_description(f"Epoch {epoch} / {num_epochs}, loss = {loss:.2f}, acc = {acc:.2f}, val_acc_ID = {val_acc[0]:.2f}, val_acc_RT = {val_acc[1]:.2f}, best_val_acc = {best_val_acc:.2f}")

        images = images.to(device)
        targets = targets.to(device)

        image_features = model(images)
        outputs = classifier(image_features)

        loss = CE_criterion(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        acc = train_acc(outputs, targets)*100

    # Validation
    val_acc_ID = test(dataloader=val_dataloader[0], model=model, classifier=classifier, 
                      test_label_map=[i for i in range(1000)], device=device)
    val_acc_RT= test(dataloader=val_dataloader[1], model=model, classifier=classifier, 
                     test_label_map=[i for i in range(1000)], device=device)
    
    # store both ID and RT val acc
    val_acc_list_ID.append(val_acc_ID/100)
    val_acc_list_RT.append(val_acc_RT/100)
    val_acc = (val_acc_ID, val_acc_RT)

    state = {}
    state['model'] = model.state_dict()
    state['head'] = classifier.state_dict()
    ckpt_name = f'model_bs{batch_size}_lr-cls{lr_cls}_lr-bkb{lr_backbone}_wd{weight_decay}_epoch_{epoch}_iter_{num_iter}.pth'
    torch.save(state, ckpt_path+ckpt_name) # Save the checkpoint

Start few-shot finetuning with RA ......


  0%|          | 0/7624 [00:00<?, ?it/s]

Epoch 1 / 10, loss = 1.52, acc = 65.62, val_acc_ID = -1.00, val_acc_RT = -1.00, best_val_acc = -1.00: 100%|██████████| 7624/7624 [15:43<00:00,  8.08it/s]
Epoch 2 / 10, loss = 1.65, acc = 67.19, val_acc_ID = 78.28, val_acc_RT = 45.65, best_val_acc = -1.00: 100%|██████████| 7624/7624 [15:44<00:00,  8.07it/s]
Epoch 3 / 10, loss = 1.52, acc = 70.31, val_acc_ID = 80.04, val_acc_RT = 46.95, best_val_acc = -1.00: 100%|██████████| 7624/7624 [15:45<00:00,  8.07it/s]
Epoch 4 / 10, loss = 1.58, acc = 68.75, val_acc_ID = 81.05, val_acc_RT = 47.92, best_val_acc = -1.00: 100%|██████████| 7624/7624 [15:44<00:00,  8.07it/s]
Epoch 5 / 10, loss = 1.47, acc = 67.19, val_acc_ID = 81.69, val_acc_RT = 48.06, best_val_acc = -1.00: 100%|██████████| 7624/7624 [15:44<00:00,  8.07it/s]
Epoch 6 / 10, loss = 1.40, acc = 65.62, val_acc_ID = 82.37, val_acc_RT = 48.40, best_val_acc = -1.00: 100%|██████████| 7624/7624 [15:44<00:00,  8.07it/s]
Epoch 7 / 10, loss = 1.39, acc = 68.75, val_acc_ID = 82.63, val_acc_RT = 48.

6. select checkpoint by $\text{gF1} = 2 \times \frac{\Delta_{ID} \times \Delta_{RT}}{\Delta_{ID} + \Delta_{RT}}$ (Eq. 2 in the paper)

In [17]:
EPS = 1e-9 # avoid dividing by zero 

val_acc_list_ID = np.array(val_acc_list_ID)
val_acc_list_RT = np.array(val_acc_list_RT)
# calculate gF1 score
delta_ID = val_acc_list_ID - val_acc_list_ID.min()
delta_RT = val_acc_list_RT - val_acc_list_RT.min()
gF1_list = 2 * delta_ID * delta_RT / (delta_ID + delta_RT + EPS) 
# select the checkpoint with the highest gF1 score
best_gF1 = max(gF1_list)
best_epoch = np.argmax(gF1_list) + 1
best_iter = best_epoch * len(train_dataloader)

print(f'Best gF1 score at stage-1: {round(best_gF1, 4)} at epoch {best_epoch}, iter {best_iter}')

Best gF1 score at stage-1: 0.0378 at epoch 10, iter 76240


7. Test the model on ID and OOD datasets

In [19]:
# load the checkpoint selected by gF1
best_model_path = ckpt_path + f'model_bs{batch_size}_lr-cls{lr_cls}_lr-bkb{lr_backbone}_wd{weight_decay}_epoch_{best_epoch}_iter_{best_iter}.pth'
checkpoint = torch.load(best_model_path) 
model.load_state_dict(checkpoint['model'])
classifier.load_state_dict(checkpoint['head'])

results_dict = {}
for test_dataset in [imagenet_test, imagenetv2_test, imagenet_sketch_test, imagenet_a_test, imagenet_r_test]:

    dataset_name = test_dataset.dataset_name
    test_dataloader = torch.utils.data.DataLoader(
        dataset=test_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=4)
    test_label_map = test_dataset.label_map
    test_acc = test(dataloader=test_dataloader, model=model, classifier=classifier, 
                    test_label_map=test_label_map, device=device)
    results_dict[dataset_name] = {'test_acc': test_acc}

ood = []
for dataset_name in results_dict.keys():
    print(f"{dataset_name} Test acc = {results_dict[dataset_name]['test_acc']}")
    if dataset_name != 'ImageNet-1k':
        ood.append(results_dict[dataset_name]['test_acc'])
avg_ood = np.mean(ood)
print(f"Avg OOD Test acc = {avg_ood}")

ImageNet-1k Test acc = 77.482
ImageNet-v2 Test acc = 71.31
ImageNet-Sketch Test acc = 59.920611527049076
ImageNet-A Test acc = 58.13333333333334
ImageNet-R Test acc = 78.32666666666667
Avg OOD Test acc = 66.92265288176226


## VEST-stage2: partial finetuning with adversarial perturbation

8. Reset dataset and dataloader

In [26]:
imagenet_train, _ = datasets.build_imagenet_few_shot_dataset_demo('imagenet', 'train', data_seed, train_preprocess,
                                                             root=root, num_shots=num_shots, w_retrival=False) # set w_retrival=False to only use the few-shot samples

train_dataloader = torch.utils.data.DataLoader(
    dataset=imagenet_train,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=False)

len(imagenet_train)

Loading few-shot data from data_resource/imagenet/fewshot16_seed1.txt.


16000

9. Reset the optimizer and learning rate scheduler

In [27]:
lr_backbone = 1e-6
lr_cls = 1e-4   # set the learning rate for the classifier
weight_decay = 0.01

# Define the optimizer
param_groups = [
            {"params": [p for name, p in model.named_parameters() if p.requires_grad], "lr": lr_backbone},
            {"params": [p for p in classifier.parameters()], "lr": lr_cls},
        ]
optimizer = torch.optim.AdamW(param_groups, lr=lr_cls, weight_decay=weight_decay, betas=(0.9, 0.999))

# Define the learning rate scheduler
num_epochs = 10
total_iter = len(train_dataloader) * num_epochs
warmup_iter = 18
warmup_lr = 1e-8
scheduler = build_lr_scheduler(optimizer,
                               lr_scheduler="cosine",
                               warmup_iter=warmup_iter,
                               max_iter=total_iter,
                               warmup_type="linear",
                               warmup_lr=warmup_lr,
                               verbose=False)

10. Define the adversarial perturbation

In [28]:
from utils.attack import PGD

eps = 0.007  # perturbation magnitude
steps = 10  # number of perturbation steps
attack = PGD(model, classifier, eps=eps, alpha=eps / 10, steps=steps)

11. Start training

In [ ]:
CE_criterion = nn.CrossEntropyLoss()

best_val_acc = -1
loss, acc = [-1] * 2
val_acc = (-1, -1)
val_acc_list_ID = []
val_acc_list_RT = []
ckpt_path = 'demo_ckpt/VEST_dinov2_top4/' # path to save checkpoints
num_iter = 0

model.to(device)
classifier.to(device)
model.train()
classifier.train()
print(f"Start few-shot finetuning with AP ......")
for epoch in range(1, num_epochs + 1):

    pbar_iter = tqdm.tqdm(train_dataloader)
    for idx, (images, targets) in enumerate(pbar_iter):
        num_iter += 1
        pbar_iter.set_description(f"Epoch {epoch} / {num_epochs}, loss = {loss:.2f}, acc = {acc:.2f}, val_acc_ID = {val_acc[0]:.2f}, val_acc_RT = {val_acc[1]:.2f}, best_val_acc = {best_val_acc:.2f}")

        images = images.to(device)
        targets = targets.to(device)

        # use forzen blocks to extract midfeatures
        image_midfeatures = model.get_midfeatures(images, k=ft_topk_blks)
        clean_features = model.get_features(image_midfeatures, k=ft_topk_blks)

        adv_midfeatures = attack(image_midfeatures, targets, k=ft_topk_blks, is_encoder=False)
        adv_features = model.get_features(adv_midfeatures, k=ft_topk_blks)

        clean_logits = classifier(clean_features)
        adv_logits = classifier(adv_features)

        clean_loss = CE_criterion(clean_logits, targets)
        adv_loss = CE_criterion(adv_logits, targets)
        loss = clean_loss + adv_loss # We employ the same weight for both clean and adv loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

    # Validation
    val_acc_ID = test(dataloader=val_dataloader[0], model=model, classifier=classifier, 
                      test_label_map=[i for i in range(1000)], device=device)
    val_acc_RT= test(dataloader=val_dataloader[1], model=model, classifier=classifier, 
                     test_label_map=[i for i in range(1000)], device=device)
    
    # store both ID and RT val acc
    val_acc_list_ID.append(val_acc_ID/100)
    val_acc_list_RT.append(val_acc_RT/100)
    val_acc = (val_acc_ID, val_acc_RT)

    state = {}
    state['model'] = model.state_dict()
    state['head'] = classifier.state_dict()
    ckpt_name = f'model_bs{batch_size}_lr-cls{lr_cls}_lr-bkb{lr_backbone}_wd{weight_decay}_epoch_{epoch}_iter_{num_iter}.pth'
    torch.save(state, ckpt_path+ckpt_name) # Save the checkpoint

Start few-shot finetuning with AP ......


Epoch 1 / 10, loss = 1.77, acc = -1.00, val_acc_ID = -1.00, val_acc_RT = -1.00, best_val_acc = -1.00: 100%|██████████| 250/250 [01:21<00:00,  3.08it/s]
Epoch 2 / 10, loss = 1.83, acc = -1.00, val_acc_ID = 85.84, val_acc_RT = 48.15, best_val_acc = -1.00: 100%|██████████| 250/250 [01:22<00:00,  3.04it/s]
Epoch 3 / 10, loss = 0.77, acc = -1.00, val_acc_ID = 87.60, val_acc_RT = 47.55, best_val_acc = -1.00: 100%|██████████| 250/250 [01:22<00:00,  3.04it/s]
Epoch 4 / 10, loss = 1.36, acc = -1.00, val_acc_ID = 88.91, val_acc_RT = 47.18, best_val_acc = -1.00: 100%|██████████| 250/250 [01:22<00:00,  3.04it/s]
Epoch 5 / 10, loss = 0.99, acc = -1.00, val_acc_ID = 89.78, val_acc_RT = 46.89, best_val_acc = -1.00: 100%|██████████| 250/250 [01:22<00:00,  3.04it/s]
Epoch 6 / 10, loss = 0.76, acc = -1.00, val_acc_ID = 90.55, val_acc_RT = 46.74, best_val_acc = -1.00: 100%|██████████| 250/250 [01:22<00:00,  3.05it/s]
Epoch 7 / 10, loss = 0.99, acc = -1.00, val_acc_ID = 91.01, val_acc_RT = 46.62, best_val

12. select checkpoint by $\text{gF1} = 2 \times \frac{\Delta_{ID} \times \Delta_{RT}}{\Delta_{ID} + \Delta_{RT}}$ (Eq. 2 in the paper)

In [30]:
EPS = 1e-9

val_acc_list_ID = np.array(val_acc_list_ID)
val_acc_list_RT = np.array(val_acc_list_RT)
# calculate gF1 score
delta_ID = val_acc_list_ID - val_acc_list_ID.min()
delta_RT = val_acc_list_RT - val_acc_list_RT.min()
gF1_list = 2 * delta_ID * delta_RT / (delta_ID + delta_RT + EPS) 
# select the checkpoint with the highest gF1 score
best_gF1 = max(gF1_list)
best_epoch = np.argmax(gF1_list) + 1
best_iter = best_epoch * len(train_dataloader)

print(f'Best gF1 score at stage-2: {round(best_gF1, 4)} at epoch {best_epoch}, iter {best_iter}')

Best gF1 score at stage-2: 0.0134 at epoch 2, iter 500


13. Test the model on ID and OOD datasets

In [32]:
best_model_path = ckpt_path + f'model_bs{batch_size}_lr-cls{lr_cls}_lr-bkb{lr_backbone}_wd{weight_decay}_epoch_{best_epoch}_iter_{best_iter}.pth'
checkpoint = torch.load(best_model_path) 
model.load_state_dict(checkpoint['model'])
classifier.load_state_dict(checkpoint['head'])

results_dict = {}
for test_dataset in [imagenet_test, imagenetv2_test, imagenet_sketch_test, imagenet_a_test, imagenet_r_test]:

    dataset_name = test_dataset.dataset_name
    test_dataloader = torch.utils.data.DataLoader(
        dataset=test_dataset,
        batch_size=64,
        shuffle=False,
        num_workers=4)
    test_label_map = test_dataset.label_map
    test_acc = test(dataloader=test_dataloader, model=model, classifier=classifier, 
                    test_label_map=test_label_map, device=device)
    results_dict[dataset_name] = {'test_acc': test_acc}

ood = []
for dataset_name in results_dict.keys():
    print(f"{dataset_name} Test acc = {results_dict[dataset_name]['test_acc']}")
    if dataset_name != 'ImageNet-1k':
        ood.append(results_dict[dataset_name]['test_acc'])
avg_ood = np.mean(ood)
print(f"Avg OOD Test acc = {avg_ood}")

ImageNet-1k Test acc = 79.30199999999999
ImageNet-v2 Test acc = 72.52
ImageNet-Sketch Test acc = 60.17999960698776
ImageNet-A Test acc = 58.89333333333333
ImageNet-R Test acc = 78.87666666666667
Avg OOD Test acc = 67.61749990174694
